Stored procedures in PL/SQL
A stored procedure is a named block of PL/SQL code stored in the Oracle database that you can execute whenever needed.

For example, instead of writing the same employee salary update query repeatedly, you can create a procedure once and call it whenever required.

Q1
our company is developing a system to manage employee 
performance bonuses. We need to calculate and update 
the performance bonus for each employee based on their **performance score**. 
the performance bonus is calculated as follows. a base bonus of 
$100,plus an additional $10 for each performance point above 80. 
if any employee performance score is 100 or higher ,
they receive an extra $200 bonus.
write a simple PL\SQL Procedure thats calculates and 
updates the performance bonus for each employee based 
pm their performance score stored in the employee table.
the procedure should use control statements to handle 
the logic for calculating the bonuses.


#Answer::::
Let's understand your PL/SQL procedure step by step, especially how the `FOR LOOP` works, how `emp` gets its values, and how the bonus is calculated and updated for each employee.

## 1. Complete code
Table_name is employee = EmployeeID, PerformanceScore, Bonus
```

CREATE OR REPLACE PROCEDURE CalculatePerformanceBonus
AS
    v_bonus NUMBER;
BEGIN
    FOR emp IN (
        SELECT EmployeeID, PerformanceScore
        FROM Employee
    )
    LOOP
        v_bonus := 100;

        IF emp.PerformanceScore >= 100 THEN
            v_bonus := v_bonus
                       + ((emp.PerformanceScore - 80) * 10)
                       + 200;

        ELSIF emp.PerformanceScore > 80 THEN
            v_bonus := v_bonus
                       + ((emp.PerformanceScore - 80) * 10);
        END IF;

        UPDATE Employee
        SET Bonus = v_bonus
        WHERE EmployeeID = emp.EmployeeID;

    END LOOP;

    COMMIT;
END;
/


=== To execute the procedure

BEGIN
Calculate-performancebonus;
END;
/


```

## 2. Understand the `FOR LOOP` first

This is the most important part of your code.

```
FOR emp IN (
    SELECT EmployeeID, PerformanceScore
    FROM Employee
)
LOOP
```

Think of this as saying:

> Retrieve the employee ID and performance score for every employee, and process one employee at a time.

Suppose your `Employee` table contains these records:

| EmployeeID | PerformanceScore | Bonus |
| ---------- | ---------------- | ----- |
| 101        | 75               | NULL  |
| 102        | 85               | NULL  |
| 103        | 100              | NULL  |

The `SELECT` query retrieves these three rows.

The loop processes them one by one:

## 1

First iteration

`emp.EmployeeID = 101`

`emp.PerformanceScore = 75`

## 2

Second iteration

`emp.EmployeeID = 102`

`emp.PerformanceScore = 85`

## 3

Third iteration

`emp.EmployeeID = 103`

`emp.PerformanceScore = 100`

After processing the third employee, the loop ends because there are no more rows to process.

### What exactly is `emp`?

`emp` is a loop record variable that Oracle creates automatically for this cursor `FOR LOOP`.

You do not need to declare it separately.

Because your query selects two columns, you can access them through that record:

- `emp.EmployeeID`
- `emp.PerformanceScore`

You can think of `emp` as a temporary record holding the current employee's values during each iteration.

Important: `emp` is not the whole Employee table. It represents just the current row being processed.

## 3. What happens inside the loop?

### Step A: Initialize the bonus

```
v_bonus := 100;
```

Every time the loop starts a new iteration, `v_bonus` is reset to 100.

For example:

- Employee 101 starts with a bonus of 100.
- Employee 102 starts with a bonus of 100.
- Employee 103 starts with a bonus of 100.

This is important because each employee's bonus must be calculated independently.

`v_bonus` is a variable declared in the procedure, whereas `emp` is the loop record automatically created by Oracle.

### Step B: Check the first condition

```
IF emp.PerformanceScore >= 100 THEN
    v_bonus := v_bonus
               + ((emp.PerformanceScore - 80) * 10)
               + 200;
```

If the score is 100 or higher, the employee gets:

1. Base bonus of 100.
2. An additional 10 for every point above 80.
3. An extra 200 for scoring at least 100.

For an employee scoring 100:

\\[ 100 + ((100-80)\times10) + 200 = 500 \\]

So `v_bonus` becomes 500.

### Step C: Check `ELSIF`

```
ELSIF emp.PerformanceScore > 80 THEN
    v_bonus := v_bonus
               + ((emp.PerformanceScore - 80) * 10);
```

`ELSIF` means "else, if."

Oracle checks this condition only when the first `IF` condition is false.

For an employee scoring 85:

\\[ 100 + ((85-80)\times10) = 150 \\]

So `v_bonus` becomes 150.

What if the score is 75?

- Score greater than or equal to 100? No.
- Score greater than 80? No.
- The bonus remains 100.

### Step D: Update the employee's bonus

```
UPDATE Employee
SET Bonus = v_bonus
WHERE EmployeeID = emp.EmployeeID;
```

This writes the calculated bonus to the matching employee's row.

For example, during the first iteration:

```
UPDATE Employee
SET Bonus = 100
WHERE EmployeeID = 101;
```

During the second iteration, conceptually:

```
UPDATE Employee
SET Bonus = 150
WHERE EmployeeID = 102;
```

During the third iteration:

```
UPDATE Employee
SET Bonus = 500
WHERE EmployeeID = 103;
```

You don't need to write these three statements yourself. The loop executes the same `UPDATE` statement with different values each time.

Why do we use `WHERE EmployeeID = emp.EmployeeID`?

Without the `WHERE` condition, the `UPDATE` could change the bonus for every employee in the table during each iteration. The condition ensures that only the current employee's row is updated.

## 4. How does the entire procedure execute?

Start procedure

Fetch the next employee using `FOR emp IN (...)`

Set `v_bonus := 100`

Check performance score using `IF / ELSIF`

Update the current employee's bonus

Repeat for the next employee until no rows remain.

`COMMIT` — save the changes

End procedure

## 5. What do `COMMIT`, `END LOOP`, and `END` mean?

```
END LOOP;

COMMIT;
END;
/
```

- `END LOOP;` — finishes the loop body and moves to the next row. When there are no more rows, execution continues after the loop.
- `COMMIT;` — saves the transaction's changes to the database.
- `END;` — finishes the PL/SQL procedure's executable block.
- `/` — in Oracle tools that support script execution, submits the completed procedure definition for compilation.

Notice that `COMMIT` is outside the loop. Therefore, the procedure commits after it finishes processing all employees, rather than committing once per employee.

## 6. Test your understanding

Question: Suppose the current employee has `EmployeeID = 104` and `PerformanceScore = 90`. What values will `emp.EmployeeID`, `v_bonus` and the final bonus have during this iteration?

`emp.EmployeeID = 104`, final bonus = 200

`emp.EmployeeID = 104`, final bonus = 100

`emp.EmployeeID = 104`, final bonus = 300

`emp.EmployeeID = 90`, final bonus = 200

Check my answer

The key concept to remember for interviews is that a cursor `FOR LOOP` fetches each query result into a record automatically, processes that row, and repeats until all rows have been handled.

# **Q2) **
our company manages a loyality program for customers. **we need to generate monthly statements that calculate
the total points earned by each customer based on their purchases**. points are awarded as follows: 
**1 point per dollar spent**. 
Additionally,if a customer **spends over $500 in a month**,they receive a **bonus of 50 points**. 
write a simple and under stable PL\SQL procedure in detail 
explaining each code step in detail that calculates and updates the total loyality points for each customer 
for a given month. 
the procedure should use control statements to handle the logic for calculating the points,including 
the bonus for spending over $500 using below customers and purchases tables as reference 
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100),
    total_points INT
);

CREATE TABLE purchases (
    purchase_id INT PRIMARY KEY,
    customer_id INT,
    purchase_amount DECIMAL(10, 2),
    purchase_date DATE,
    CONSTRAINT fk_customer
        FOREIGN KEY (customer_id)
        REFERENCES customers (customer_id)
);
- give me best performance oriented simple and also alternative solutions for the above queston?

#Answer ::::
For this problem, we can write a PL/SQL stored procedure that calculates loyalty points for each customer for a given month, adds a 50-point bonus when monthly spending exceeds $500, and updates the `totalpoints` column in the `customers` table.

I'll show you the simple, beginner-friendly solution first, followed by a more performance-oriented alternative.

## 1. Understand the business logic

The rules are:

- Earn 1 point for every $1 spent during the given month.
- If the customer's total monthly spending is strictly greater than $500, award an additional 50 points.
- Update the customer's `totalpoints` column.

For example:

| Customer | Monthly spending | Base points | Bonus | Total points earned |
| -------- | ---------------- | ----------- | ----- | ------------------- |
| Rahul    | $300             | 300         | 0     | 300                 |
| Priya    | $500             | 500         | 0     | 500                 |
| Amit     | $650             | 650         | 50    | 700                 |
| Sneha    | $800             | 800         | 50    | 850                 |

I'm assuming that `totalpoints` means the points earned for the specified month, so the procedure replaces the existing value. If you want to accumulate points across months, I'll show that adjustment too.

## 2. Create the tables in Oracle

Your table definitions need a small Oracle syntax adjustment: use `NUMBER(10,2)` for decimal amounts and `purchase_amount` as the column name.

```
CREATE TABLE customers (
    customerid   NUMBER PRIMARY KEY,
    customername VARCHAR2(100),
    totalpoints  NUMBER DEFAULT 0
);

CREATE TABLE purchases (
    purchaseid     NUMBER PRIMARY KEY,
    customerid     NUMBER,
    purchase_amount NUMBER(10,2),
    purchasedate   DATE,
    CONSTRAINT fk_customer
        FOREIGN KEY (customerid)
        REFERENCES customers(customerid)
);
```

## 3. Solution 1 — Simple PL/SQL procedure using a FOR LOOP and IF

This solution is best for learning PL/SQL control statements. It calculates each customer's monthly spending, applies the bonus condition, and updates their points.

```
================================ Use this ===============================================


CREATE OR REPLACE PROCEDURE CalculateLoyaltyPoints
(
    p_month IN DATE
)
AS
    v_month_start DATE;
    v_month_end   DATE;
    v_points      NUMBER;
BEGIN

    -- First day of the given month
    v_month_start := TRUNC(p_month, 'MM');

    -- First day of next month
    v_month_end := ADD_MONTHS(v_month_start, 1);

    -- Calculate monthly spending for each customer
    FOR cust IN
    (
        SELECT c.customerid,
               NVL(SUM(p.purchase_amount), 0) AS monthly_spending
        FROM customers c
        LEFT JOIN purchases p
               ON c.customerid = p.customerid
              AND p.purchasedate >= v_month_start
              AND p.purchasedate < v_month_end
        GROUP BY c.customerid
    )
    LOOP

        -- 1 point per dollar spent
        v_points := TRUNC(cust.monthly_spending);

        -- Additional 50 points if spending is greater than 500
        IF cust.monthly_spending > 500 THEN
            v_points := v_points + 50;
        END IF;

        -- Update customer points
        UPDATE customers
        SET totalpoints = v_points
        WHERE customerid = cust.customerid;

        -- Display result
        DBMS_OUTPUT.PUT_LINE(
            'Customer ID: ' || cust.customerid ||
            ' Spending: ' || cust.monthly_spending ||
            ' Points: ' || v_points
        );

    END LOOP;

    COMMIT;

    DBMS_OUTPUT.PUT_LINE(
        'Loyalty Points Updated Successfully'
    );

END;
/

ORRRRRRRRRRRRRRRRRRRRRRRRRRRRRRR dont use this =========================

CREATE OR REPLACE PROCEDURE CalculateLoyaltyPoints (
    p_month IN DATE
)
AS
    v_month_start DATE;
    v_month_end   DATE;
    v_spending    NUMBER;
    v_points      NUMBER;
BEGIN
    -- Get the first day of the requested month
    v_month_start := TRUNC(p_month, 'MM');

    -- Get the first day of the next month
    v_month_end := ADD_MONTHS(v_month_start, 1);

    -- Process each customer
    FOR cust IN (
        SELECT c.customerid,
               NVL(SUM(p.purchase_amount), 0) AS monthly_spending
        FROM customers c
        LEFT JOIN purchases p
            ON c.customerid = p.customerid
           AND p.purchasedate >= v_month_start
           AND p.purchasedate < v_month_end
        GROUP BY c.customerid
    )
    LOOP
        -- Get monthly spending
        v_spending := cust.monthly_spending;

        -- One point per whole dollar spent
        v_points := FLOOR(v_spending);

        -- Bonus points for spending over $500
        IF v_spending > 500 THEN
            v_points := v_points + 50;
        END IF;

        -- Update the customer's points
        UPDATE customers
        SET totalpoints = v_points
        WHERE customerid = cust.customerid;

    END LOOP;

    COMMIT;
END;
/

```

### How the procedure works

Step 1 — Accept the month

```
p_month IN DATE
```

The procedure accepts a date representing the month you want to process. For example, you can pass any date in March 2026.

Step 2 — Define the date range

```
v_month_start := TRUNC(p_month, 'MM');
v_month_end := ADD_MONTHS(v_month_start, 1);
```

If the input is March 15, 2026:

- `v_month_start` becomes March 1, 2026.
- `v_month_end` becomes April 1, 2026.

We use a half-open date range, including the start and excluding the end, so purchases on March 31 are included even if they have a time component. This also avoids applying a function to the `purchasedate` column in the filter, which can help Oracle use a suitable index.&#x20;

[image](https://www.google.com/s2/favicons?domain=https://asktom.oracle.com\&sz=32)

Oracle Magazine

+1



Step 3 — Calculate monthly spending

```
FOR cust IN (
    SELECT c.customerid,
           NVL(SUM(p.purchase_amount), 0) AS monthly_spending
    FROM customers c
    LEFT JOIN purchases p
        ON c.customerid = p.customerid
       AND p.purchasedate >= v_month_start
       AND p.purchasedate < v_month_end
    GROUP BY c.customerid
)
LOOP
```

This query calculates total spending for every customer during the requested month.

- `SUM(p.purchase_amount)` adds all qualifying purchases.
- `NVL(..., 0)` returns zero if the customer has no purchases.
- `LEFT JOIN` ensures customers without purchases are included.
- `GROUP BY c.customerid` produces one result per customer.
- `FOR cust IN (...) LOOP` processes those results one at a time.

Step 4 — Calculate the base points

```
v_points := TRUNC(v_spending);
```

We use `TRUNC` because `totalpoints` is a whole-number column and the rule awards one point per whole dollar. For example, $250.75 earns 250 base points under this assumption.

Step 5 — Apply the bonus using `IF`

```
IF v_spending > 500 THEN
    v_points := v_points + 50;
END IF;
```

The condition must be strictly greater than $500.

- Spending of $500 earns no bonus.
- Spending of $500.01 earns the additional 50 points.

Step 6 — Update the customer

```
UPDATE customers
SET totalpoints = v_points
WHERE customerid = cust.customerid;
```

Only the current customer's record is updated.

Step 7 — Commit

```
COMMIT;
```

Saves all updates after the loop has processed the customers.

### Execute the procedure

```
BEGIN
    CalculateLoyaltyPoints(DATE '2026-03-01');
END;
/
```

Check the results:

```
SELECT customerid, customername, totalpoints
FROM customers
ORDER BY customerid;
```

## 4. Solution 2 — Performance-oriented alternative

For a large number of customers, you can avoid the PL/SQL `FOR LOOP` and individual `UPDATE` statements by using a single set-based `UPDATE` statement.

This version uses SQL's `CASE` expression instead of PL/SQL's `IF` statement.

```

CREATE OR REPLACE PROCEDURE CalculateLoyaltyPoints_Fast (
    p_month IN DATE
)
AS
    v_month_start DATE;
    v_month_end   DATE;
BEGIN
    v_month_start := TRUNC(p_month, 'MM');
    v_month_end   := ADD_MONTHS(v_month_start, 1);

    UPDATE customers c
    SET totalpoints = (
        SELECT TRUNC(NVL(SUM(p.purchase_amount), 0))
               + CASE
                   WHEN NVL(SUM(p.purchase_amount), 0) > 500
                   THEN 50
                   ELSE 0
                 END
        FROM purchases p
        WHERE p.customerid = c.customerid
          AND p.purchasedate >= v_month_start
          AND p.purchasedate < v_month_end
    );

    COMMIT;
END;
/

```

Execute it in the same way:

```
BEGIN
    CalculateLoyaltyPoints_Fast(DATE '2026-03-01');
END;
/
```

### Why is this alternative useful?

- It uses one SQL `UPDATE` statement instead of issuing an update for every customer from a PL/SQL loop.
- `SUM()` calculates the customer's monthly spending.
- `CASE` applies the additional 50 points when spending exceeds $500.
- Customers without purchases receive zero points.
- Oracle can optimize the set-based statement, although actual performance depends on table size, indexes, and the execution plan.

For a large dataset, consider an index on `purchases(customerid, purchasedate)` and evaluate the execution plan against your real data. The best index depends on the workload.

## 5. Compare the two solutions

| Feature                     | Solution 1: Loop + IF                 | Solution 2: Single UPDATE          |
| --------------------------- | ------------------------------------- | ---------------------------------- |
| Bonus logic                 | PL/SQL `IF`                           | SQL `CASE`                         |
| Updates                     | One per customer                      | One set-based statement            |
| Easy for beginners          | Yes                                   | Yes, once SQL is familiar          |
| Large datasets              | May incur more PL/SQL-to-SQL overhead | Often a better starting point      |
| Control statements practice | Excellent                             | Demonstrates SQL conditional logic |

My recommendation: Learn Solution 1 first because it demonstrates the PL/SQL control statements in your question. For production-scale processing, start by evaluating Solution 2.

One final business-rule distinction: both procedures replace `totalpoints` with the points earned for the selected month. If `totalpoints` is intended to be a lifetime balance, you should instead add the newly calculated monthly points to the existing balance, while ensuring the same month cannot be credited twice.

note: DEFAULT 0 means: if you don’t give a value for that column when inserting a row, 
Oracle automatically stores 0.
NUMBER(10,2) means a number with up to 10 digits in total, of which 2 are after the decimal point.
10 = precision: the total number of digits allowed.
2 = scale: how many of those digits come after the decimal point.
NUMBER(10,2) means a number with up to 10 digits in total, of which 2 are after the decimal point.

10 = precision: the total number of digits allowed.
2 = scale: how many of those digits come after the decimal point.

**TRUNC** is an Oracle function that **cuts off (truncates) part of a value** 
instead of rounding it. It works on both dates and numbers.

## 1. TRUNC with dates

It chops a date back to a chosen unit, and the rest is reset.

```sql
TRUNC(date)            -- removes the time, keeps the day
TRUNC(date, 'MM')      -- first day of the month
TRUNC(date, 'YYYY')    -- first day of the year
```

| Expression | Result |
|---|---|
| `TRUNC(SYSDATE)` | Today at 00:00:00 |
| `TRUNC(DATE '2025-08-15', 'MM')` | `01-AUG-2025` |
| `TRUNC(DATE '2025-08-15', 'YYYY')` | `01-JAN-2025` |

## 2. TRUNC with numbers

It drops digits after a given decimal place **without rounding**.

```sql
TRUNC(number, places)
```

| Expression | Result |
|---|---|
| `TRUNC(125.789)` | `125` |
| `TRUNC(125.789, 2)` | `125.78` |
| `TRUNC(125.789, 1)` | `125.7` |
| `TRUNC(125.789, -1)` | `120` |

## TRUNC vs ROUND

- `TRUNC(125.789, 2)` gives `125.78` (just cuts off)
- `ROUND(125.789, 2)` gives `125.79` (rounds up)

In your line `TRUNC(p_month, 'MM')`, it's the date version: it cuts the date back to 
the first day of that month.

`ADD_MONTHS(date, n)` is an Oracle function that **adds `n` months to a date** and returns the new date.

In your line, `ADD_MONTHS(v_month_start, 1)` adds 1 month to `v_month_start`, so you get the **first 
day of the next month** (because `v_month_start` is already the 1st).

**Examples:**

| Expression | Result |
|---|---|
| `ADD_MONTHS(DATE '2025-08-01', 1)` | `01-SEP-2025` |
| `ADD_MONTHS(DATE '2025-12-01', 1)` | `01-JAN-2026` |
| `ADD_MONTHS(DATE '2025-08-01', -1)` | `01-JUL-2025` (negative goes back) |
| `ADD_MONTHS(DATE '2025-01-31', 1)` | `28-FEB-2025` (clamps to the last day of the month) |

**Why it's useful:** it handles different month lengths (28, 30, 31 days) and year changes for you, 
so you don't have to calculate them yourself.

**Together with TRUNC:**

```sql
v_month_start := TRUNC(p_month, 'MM');           -- 01-AUG-2025
v_month_end   := ADD_MONTHS(v_month_start, 1);   -- 01-SEP-2025
```

Then you can select everything from August with:

```sql
WHERE purchase_date >= v_month_start
  AND purchase_date <  v_month_end
```

**Note:** `ADD_MONTHS(...)` by itself, ending with `;`, is not a complete statement. 
It must be assigned to a variable (`v_month_end := ...;`) or used inside a 
`SELECT`, like `SELECT ADD_MONTHS(SYSDATE, 1) FROM dual;`.

Why p_month IN DATE?

p_month is an input parameter of DATE datatype used to specify the month for which loyalty points should be calculated.

Why call using BEGIN...END?

Procedures are executed inside a PL/SQL block. BEGIN...END starts and ends the block, and / tells Oracle to run it.

Also remember:

SQL
IN -> Input only
OUT -> Output only
IN OUT -> Input + Output

# Q3 Loyalty points Procedure: 
Create a procedure to calculate and update loyalty points for customers based on their 
**monthly purchase amount**. There are two tables: a **Customer table** containing the 
customer's loyalty points and a **Transactions table** containing the customer's purchase 
transactions across multiple dates. The procedure should **group the transactions** by 
customer and month, calculate the total purchase amount for each month, and if the 
**monthly total purchase amount is greater than 500**, award the customer 100 loyalty points. 
The calculated loyalty points should then be updated in the Customer table. 

Yes, I checked the requirement again carefully.

Requirement::::

Group transactions by customer and month.

Calculate total monthly purchase amount.

If monthly purchase > 500, award 100 points.

Update Customer table.

Use PL/SQL control statements.

#### Recommended Final Answer

CREATE OR REPLACE PROCEDURE Calculate_Loyalty_Points
(
    p_month IN DATE
)
AS
    v_points NUMBER;
    v_month_start DATE;
    v_month_end   DATE;

BEGIN

    -- First day of the month
    v_month_start := TRUNC(p_month,'MM');

    -- First day of next month
    v_month_end := ADD_MONTHS(v_month_start,1);

    -- Calculate monthly purchases customer-wise
    FOR rec IN
    (
        SELECT customerid,
               SUM(purchase_amount) AS total_purchase
        FROM transactions
        WHERE purchasedate >= v_month_start
          AND purchasedate < v_month_end
        GROUP BY customerid
    )
    LOOP

        -- Loyalty points calculation
        IF rec.total_purchase > 500 THEN
            v_points := 100;
        ELSE
            v_points := 0;
        END IF;

        -- Update customer table
        UPDATE customers
        SET totalpoints = v_points
        WHERE customerid = rec.customerid;

    END LOOP;

    COMMIT;

END;
/

Why this is correct
1. Procedure Parameter
p_month IN DATE


✅ Better than:

p_month NUMBER,
p_year NUMBER


because one parameter contains both month and year.

2. Month Range
v_month_start := TRUNC(p_month,'MM');
v_month_end := ADD_MONTHS(v_month_start,1);


If input is:

DATE '2026-07-15'


then:

v_month_start = 01-JUL-2026
v_month_end   = 01-AUG-2026


✅ Correct

3. Group By
GROUP BY customerid


Question says:

group transactions by customer and month

Since we're already filtering for a single month using:

WHERE purchasedate >= v_month_start
AND purchasedate < v_month_end


grouping by:

customerid


is sufficient.

✅ Correct

4. Control Statement
IF rec.total_purchase > 500 THEN


✅ Uses IF condition as requested.

5. Update Customer Table
UPDATE customers
SET totalpoints = v_points
WHERE customerid = rec.customerid;


✅ Matches requirement.

One Interview Point

If interviewer asks:

Why not GROUP BY month also?

Answer:

Because the procedure processes one month at a time.
The WHERE clause already filters a single month, so GROUP BY customerid is enough.
If we were processing multiple months together, then we would GROUP BY customerid and month.


Example:

GROUP BY customerid,
         EXTRACT(MONTH FROM purchasedate),
         EXTRACT(YEAR FROM purchasedate)


That would be needed only when calculating multiple months in one query.

Verdict

✅ Syntax: Correct
 ✅ Logic: Correct
 ✅ Uses Control Statement: Yes (IF)
 ✅ Uses Grouping: Yes (GROUP BY customerid)
 ✅ Interview Ready: Yes

This is the version I would write in an Oracle PL/SQL interview.

Below are both approaches completely, along with when to use each.

Approach 1: Explicit Cursor + SELECT SUM() Inside Loop

This is easier for beginners because it clearly shows:

Cursor
FOR LOOP
SELECT INTO
IF Condition
UPDATE
CREATE OR REPLACE PROCEDURE Calculate_Loyalty_Points
(
    p_month IN NUMBER,
    p_year  IN NUMBER
)
IS

    v_total_purchase NUMBER;
    v_points         NUMBER;

    CURSOR c_customer IS
        SELECT customerid
        FROM customers;

BEGIN

    FOR rec IN c_customer LOOP

        -- Calculate monthly purchase amount
        SELECT NVL(SUM(purchase_amount), 0)
        INTO v_total_purchase
        FROM purchases
        WHERE customerid = rec.customerid
          AND EXTRACT(MONTH FROM purchasedate) = p_month
          AND EXTRACT(YEAR  FROM purchasedate) = p_year;

        -- Loyalty points logic
        IF v_total_purchase > 500 THEN
            v_points := 100;
        ELSE
            v_points := 0;
        END IF;

        -- Update customer table
        UPDATE customers
        SET totalpoints = v_points
        WHERE customerid = rec.customerid;

        DBMS_OUTPUT.PUT_LINE(
            'Customer: ' || rec.customerid ||
            ' Purchase: ' || v_total_purchase ||
            ' Points: ' || v_points
        );

    END LOOP;

    COMMIT;

END;
/

Execution
BEGIN
    Calculate_Loyalty_Points(7, 2026);
END;
/

How It Works
Step 1

Cursor fetches customers.

CURSOR c_customer IS
SELECT customerid
FROM customers;


Example:

1
2
3

Step 2

For each customer:

FOR rec IN c_customer LOOP

Step 3

Calculate monthly purchase amount.

SELECT NVL(SUM(purchase_amount),0)
INTO v_total_purchase


Suppose Customer 1 purchased:

300
250


Total:

550

Step 4

Apply business rule.

IF v_total_purchase > 500 THEN
   v_points := 100;
ELSE
   v_points := 0;
END IF;


Result:

550 -> 100 Points
200 -> 0 Points

Step 5

Update customer table.

UPDATE customers
SET totalpoints = v_points
WHERE customerid = rec.customerid;

Approach 2: GROUP BY (Recommended for Interview)

This directly satisfies:

"Group transactions by customer and month"

CREATE OR REPLACE PROCEDURE Calculate_Loyalty_Points
(
    p_month IN NUMBER,
    p_year  IN NUMBER
)
IS
    v_points NUMBER;
BEGIN

    FOR rec IN
    (
        SELECT customerid,
               SUM(purchase_amount) AS total_purchase
        FROM purchases
        WHERE EXTRACT(MONTH FROM purchasedate) = p_month
          AND EXTRACT(YEAR  FROM purchasedate) = p_year
        GROUP BY customerid
    )
    LOOP

        IF rec.total_purchase > 500 THEN
            v_points := 100;
        ELSE
            v_points := 0;
        END IF;

        UPDATE customers
        SET totalpoints = v_points
        WHERE customerid = rec.customerid;

        DBMS_OUTPUT.PUT_LINE(
            'Customer: ' || rec.customerid ||
            ' Purchase: ' || rec.total_purchase ||
            ' Points: ' || v_points
        );

    END LOOP;

    COMMIT;

END;
/

Execution
BEGIN
    Calculate_Loyalty_Points(7, 2026);
END;
/

Example Data
Purchases
CustomerID   PurchaseAmount
----------   --------------
1            300
1            250
2            200
3            400
3            200

After GROUP BY
SELECT customerid,
       SUM(purchase_amount)
FROM purchases
GROUP BY customerid;


Result:

CustomerID   Total_Purchase
----------   --------------
1            550
2            200
3            600


Then loop processes:

Customer 1 -> 100 Points
Customer 2 -> 0 Points
Customer 3 -> 100 Points

Which One Should You Use?
If Interviewer Wants PL/SQL Concepts

Use Approach 1

Because it demonstrates:

Cursor
Select Into
Variables
IF
Loop
Update

If Interviewer Says

Group transactions by customer and month

Use Approach 2

Because it explicitly uses:

GROUP BY customerid


and directly matches the requirement.

Interview Answer

If asked:

Which approach is better?

Answer:

"Both solutions work. The cursor-based solution is easier for learning PL/SQL because it demonstrates cursor handling and SELECT INTO. The GROUP BY approach is generally preferred because it aggregates data first and avoids executing a SUM query for each customer, making it simpler and more efficient." ✅

NOTE:
Q: Can we use AS instead of IS in a procedure?

Answer:
Yes. In Oracle PL/SQL, both IS and AS are functionally equivalent when defining procedures, functions, packages, and triggers. There is no performance or behavioral difference. IS is more commonly used by convention in PL/SQL program units, while AS is also accepted by Oracle.

What should you remember?
Q2
Customer + Purchase

Preferred:
LEFT JOIN

Reason:
Include customers with no purchases.

Q3
Group Transactions

Preferred:
GROUP BY customerid

Reason:
Question focuses on transaction aggregation.